In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [55]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!curl -L "$data" -o data-week-4.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0   444k      0 --:--:-- --:--:-- --:--:--  444k


In [56]:
df = pd.read_csv("data-week-4.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [57]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [58]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
for c in categorical:
    df[c] = df[c].fillna('NA')
for c in numerical:
    df[c] = df[c].fillna(0)

In [59]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

df_train = df_train.reset_index(drop = True)
df_val = df_val.reset_index(drop = True)
df_test = df_test
.reset_index(drop = True)

SyntaxError: invalid syntax (2715443072.py, line 13)

In [ ]:
from sklearn.metrics import roc_auc_score

data = []
for c in numerical:
    auc = roc_auc_score(df_train['converted'], df_train[c])
    if auc < 0.5:
        auc = roc_auc_score(df_train['converted'], -df_train[c])
    else:
        data.append((c, auc))
print(sorted(data, key=lambda x: x[1], reverse=True)[0])

('lead_score', np.float64(0.7882254810906102))


In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

X = df_train[categorical + numerical].to_dict(orient = 'records')
X_val_dict = df_val[categorical + numerical].to_dict(orient = 'records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(X)
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(X_train, y_train)
X_val = dv.transform(X_val_dict)
y_pred = model.predict_proba(X_val)[:,1]
roc_auc_score(y_val, y_pred)

/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


np.float64(0.7318222288173319)

In [73]:
thresholds = np.linspace(0, 1, 101)
precisions = []
f1_data = []
diff = []
for c in thresholds:
    Actual_positive = (y_val == 1)
    Actual_negative = (y_val == 0)

    predict_positive = (y_pred >= c)
    predict_negative = (y_pred < c)

    tp = (Actual_positive & predict_positive).sum()
    fp = (Actual_negative & predict_positive).sum()
    tn = (Actual_negative & predict_negative).sum()
    fn = (Actual_positive & predict_negative).sum()

    
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * (precision * recall) / (precision + recall)
    precisions.append(precision)
    f1_data.append([c, f1])
    if precision + recall > 0:
        diff.append([c, abs(precision - recall)])

print(min(diff, key=lambda x: x[1]))
print(max(f1_data, key=lambda x: x[1]))
# import matplotlib.pyplot as plt
# plt.plot(precisions, thresholds, label = 'precision')
# plt.plot(recalls, thresholds, label = 'recall')
# plt.legend()

[np.float64(0.63), np.float64(0.007502647993409384)]
[np.float64(0.41000000000000003), np.float64(0.7576158940397352)]


/var/folders/_4/4tb_1w6x75q_qf4ng16vxs5m0000gn/T/ipykernel_72547/481138595.py:18: RuntimeWarning: invalid value encountered in scalar divide
  precision = tp / (tp + fp)


In [80]:
from sklearn.model_selection import KFold
deviation = []
def train(df_train, y_train, C = 1):
    dv = DictVectorizer(sparse=False)
    X_dict = df_train[categorical + numerical].to_dict(orient = 'records')
    X_train = dv.fit_transform(X_dict)
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000)
    model.fit(X_train, y_train)
    return dv, model
def prediction(df_val, dv, model):
    X_dict = df_val[categorical + numerical].to_dict(orient = 'records')
    X_val = dv.transform(X_dict)
    y_pred = model.predict_proba(X_val)[:, 1]
    return y_pred
for c in [0.000001, 0.001, 1]:
    aucs = []
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    for train_inx, val_inx in kfold.split(df_full_train):
        df_train = df_full_train.iloc[train_inx]
        df_val = df_full_train.iloc[val_inx]

        y_train = df_train.converted.values
        y_val = df_val.converted.values

        dv, model = train(df_train, y_train, C = c)
        y_pred = prediction(df_val, dv, model)
        auc = roc_auc_score(y_val, y_pred)
        aucs.append(roc_auc_score(y_val, y_pred))
        deviation.append(roc_auc_score(y_val, y_pred))
    print(np.mean(aucs), np.std(aucs))


/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: Run

0.6167342250714795 0.018669799234831475
0.7491721776030379 0.009996375455700753
0.7317269710972524 0.006853233151022138


/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/bulat/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: Run